# 13 - Managed ingestion design and capstone assessment

Prerequisites / execution context: Core notebooks 00-12. Managed connectors and external clients additionally require source access, network connectivity, connector-specific permissions and supported workspace features.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R13: Lakeflow Connect concepts](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/); [R14: Managed database connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/cdc-overview); [R15: Query-based connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/query-based-overview); [R16: Spark data sources](https://docs.databricks.com/aws/en/connect/spark-data-sources); [R17: Secret management](https://docs.databricks.com/aws/en/security/secrets/); [R70: Lakehouse Federation](https://docs.databricks.com/aws/en/query-federation/).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Choose an ingestion approach
Design four workloads: append-only cloud JSON; SQL Server with required deletes; daily SaaS exports; ad hoc live Teradata queries. Suggested starting points: Auto Loader; a supported managed CDC connector; a supported managed SaaS connector or file ingestion; Lakehouse Federation. Confirm the source-specific connector support and release state instead of assuming all databases are interchangeable. Official Databricks documentation: [R10: Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader); [R13: Lakeflow Connect concepts](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/); [R14: Managed database connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/cdc-overview); [R70: Lakehouse Federation](https://docs.databricks.com/aws/en/query-federation/).


## 2. Managed connector walkthrough
With an approved disposable source, select a supported connector in the ingestion UI. Follow its linked source-setup instructions. Create/use the UC connection; choose source objects and UC destination; configure the gateway/staging components if required by that connector; run initial ingestion, then an update and delete if CDC is supported. Compare initial rows and subsequent target changes. Inspect pipeline status, permissions, and refresh cadence. Record source load and recovery behavior. If no external source is available, document these steps and the required access; the local synthetic labs do not simulate managed ingestion. Official Databricks documentation: [R13: Lakeflow Connect concepts](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/); [R14: Managed database connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/cdc-overview); [R15: Query-based connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/query-based-overview).


## 3. JDBC/ODBC/REST design exercise
Use supported Spark data sources or a UC connection for JDBC ingestion; verify driver/network support and control read parallelism to avoid overwhelming the source. ODBC/JDBC clients connecting TO Databricks for queries differ from Spark reading FROM a database. A custom REST client needs pagination, retry/backoff, rate-limit handling, a durable extraction cursor, secrets, and a replay-safe landing contract. The next cell uses an offline response fixture to practice landing; it does not claim to test a live service. Official Databricks documentation: [R16: Spark data sources](https://docs.databricks.com/aws/en/connect/spark-data-sources); [R17: Secret management](https://docs.databricks.com/aws/en/security/secrets/).


In [ ]:
import json,uuid
# A deterministic stand-in for two pages returned by an authorized API client.
pages=[{"items":[{"id":"a","value":10}],"next":"page2"},
       {"items":[{"id":"b","value":20}],"next":None}]
run_id=uuid.uuid4().hex[:10]
path=f"{VOLUME}/rest_fixture/{run_id}"
for n,page in enumerate(pages):
    dbutils.fs.put(f"{path}/page_{n}.json","\n".join(json.dumps(x) for x in page["items"]),True)
landed=spark.read.schema("id STRING,value LONG").json(path)
assert landed.count()==2


### Optional JDBC ingestion with a governed connection
Requires an administrator-created JDBC UC connection and an approved disposable source table. Edit the connection name and query for your source. This cell copies rows into a managed Delta table; it is not federation alone. It is disabled by default. [Official Spark data sources](https://docs.databricks.com/aws/en/connect/spark-data-sources).


In [ ]:
RUN_JDBC=False
if RUN_JDBC:
    CONNECTION="approved_lab_jdbc_connection"
    # Adapt quoted identifiers and column names to the source database.
    QUERY="SELECT id, name FROM approved_lab_source"
    extracted=(spark.read.format("jdbc")
               .option("databricks.connection",CONNECTION)
               .option("query",QUERY).load())
    assert {"id","name"} <= set(extracted.columns)
    TARGET=f"{BRONZE}.jdbc_snapshot"
    extracted.write.format("delta").mode("overwrite").saveAsTable(TARGET)
    spark.table(TARGET).show(10,truncate=False)
    # A later incremental version needs a durable cursor/CDC contract;
    # repeated snapshots are not an update/delete-aware replication design.


### Optional local ODBC client
Run this on a local development machine with the Databricks ODBC driver and pyodbc installed and an authenticated DSN configured, following the official setup. This direction queries Databricks FROM a client. Use your driver's approved OAuth setup; do not paste tokens into this notebook. [Official pyodbc tutorial](https://docs.databricks.com/aws/en/dev-tools/pyodbc).


In [ ]:
RUN_LOCAL_ODBC=False
if RUN_LOCAL_ODBC:
    import os,re,pyodbc
    # DSN is a configured non-secret name, not a password.
    dsn=os.environ["DATABRICKS_LAB_DSN"]
    assert re.fullmatch(r"[A-Za-z0-9_-]+",dsn)
    with pyodbc.connect(f"DSN={dsn}",autocommit=True) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT current_catalog(), current_schema()")
            print(cursor.fetchone())


## 4. Capstone: rebuild from requirements
Without reading solutions: ingest eight seed events; quarantine two invalid amounts; produce four latest valid orders; retain two unmatched customers with a left join; publish total revenue 220.00; prove rerun behavior; choose a data-driven trigger; demonstrate both job branches; deploy dev/test targets; document reader access and masked fields; diagnose a provided skew scenario. Submit run IDs, assertions, a grain definition, lineage evidence, and a recovery plan. Official Databricks documentation: [R06: Medallion architecture](https://docs.databricks.com/aws/en/lakehouse/medallion); [R37: Lakeflow Jobs](https://docs.databricks.com/aws/en/jobs/); [R46: Declarative Automation Bundles](https://docs.databricks.com/aws/en/dev-tools/bundles/); [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference).


## Evaluation rubric (author-designed)
Score 0/1/2 for each: correct result, explanation, rerun proof, failure diagnosis, permission design, environment separation, observability, and recovery. A suggested practice goal is 14/16 with no zero in correctness or security. This is a learning rubric, not an official passing score. Review STUDY_GUIDE.md for the objective matrix and scenario answer key.
